# SimCLR pretraining for YOLO12

This tutorial pretrains a **YOLO12** backbone with **SimCLR** on the football player detection
dataset, then transfers the learned backbone into a YOLO12 detector. Object labels are never
read during pretraining.

**Audience**

Students who know basic Python and have seen a PyTorch training loop.

**Learning goals**

- Prepare the Kaggle football dataset for label-free learning
- Configure SimCLR against a YOLO12 backbone
- Train on two NVIDIA T4 GPUs
- Read the training history and run manifest
- Transfer the pretrained backbone into a YOLO12 detector checkpoint

## How SimCLR learns

SimCLR builds two independently augmented views of every image. The encoder and projection
head map each view to a normalized vector, and cosine similarity $s_{i,j}=z_i^{T}z_j$ scores
each pair.

For a positive pair $(i,j)$ the NT-Xent loss is
$\ell_{i,j}=-\log\frac{\exp(s_{i,j}/\tau)}{\sum_{k\neq i}\exp(s_{i,k}/\tau)}$, where $\tau$ is
the temperature. The positive pair is pulled together while every other image in the batch acts
as a negative.

**What this means for batch size.** Negatives come from the batch, so SimCLR benefits from a
large per-GPU batch. Gradient accumulation raises the optimizer batch but does *not* add more
simultaneous negatives.

## Notebook roadmap

1. Configure the Kaggle runtime
2. Find and inspect the dataset
3. Configure SimCLR for YOLO12
4. Inspect the augmented views
5. Train on T4 x2
6. Review the training loss
7. Transfer the backbone into a YOLO12 detector
8. Try an exercise

## 1. Configure the Kaggle runtime

Open **Notebook options**, select **GPU T4 x2**, and enable Internet access. Add the
`iasadpanwhar/football-player-detection-yolov8` dataset through **Add Input**. The notebook reads
the mounted files directly and does not call the Kaggle competition API.

In [ ]:
%pip install -q --upgrade "git+https://github.com/rifat963/ssl-detection-lab.git@main" scikit-learn seaborn

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from ultralytics import YOLO

import ssldet
from ssldet import PretrainConfig, launch_distributed_pretrain
from ssldet.data import IMAGENET_MEAN, IMAGENET_STD, UnlabeledImageDataset, build_transform
from ssldet.downstream import transfer_ssl_backbone_to_yolo

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GPU_COUNT = torch.cuda.device_count()

pd.Series({
    "ssldet version": ssldet.__version__,
    "torch": torch.__version__,
    "CUDA available": torch.cuda.is_available(),
    "GPU count": GPU_COUNT,
})

A T4 x2 session should report two GPUs. Training still runs with a different GPU count, but the effective batch and runtime will change.

## 2. Find and inspect the dataset

The first path matches the full path supplied with the dataset. The second covers Kaggle's
shorter mounted-input layout.

In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]

DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.exists()), None)

if DATASET_ROOT is None:
    mounted_inputs = sorted(str(path) for path in Path("/kaggle/input").glob("*"))
    raise FileNotFoundError(
        "Attach the football-player-detection-yolov8 dataset with Add Input. "
        f"Mounted inputs: {mounted_inputs}"
    )

SPLIT_PATHS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

train_images = image_files(SPLIT_PATHS["train"]["images"])

pd.Series({
    "dataset root": str(DATASET_ROOT),
    "train images": len(train_images),
    "valid images": len(image_files(SPLIT_PATHS["valid"]["images"])),
    "test images": len(image_files(SPLIT_PATHS["test"]["images"])),
})

SimCLR reads `train/images` only. No annotation path is passed to the trainer.

## 3. Configure SimCLR for YOLO12

**Use `yolo12n.yaml`, not `yolov12n.yaml`.** Ultralytics dropped the `v` from YOLO11 onward, so
the file is named `yolo12n.yaml`. Passing `yolov12n.yaml` raises `FileNotFoundError`, and passing
a name like `yolov12n.pt` makes the reporting layer fall back to the generic
*Custom Ultralytics YOLO* family instead of *YOLO12*.

The `.yaml` suffix initializes the architecture **without** COCO weights, giving a strict
label-free run. Switch to `yolo12n.pt` to start from supervised COCO weights and adapt to
football images instead.

Scale the backbone by changing the letter: `yolo12n`, `yolo12s`, `yolo12m`, `yolo12l`, `yolo12x`.

The projection head is discarded at transfer time; only the backbone moves to the detector.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working/simclr_yolo12_football")
EPOCHS = 10
PER_GPU_BATCH_SIZE = 32

config = PretrainConfig(
    method="simclr",
    image_roots=[str(SPLIT_PATHS["train"]["images"])],
    output_dir=str(OUTPUT_DIR),
    yolo_model="yolo12n.yaml",
    epochs=EPOCHS,
    batch_size=PER_GPU_BATCH_SIZE,
    image_size=224,
    workers=2,
    max_images=None,
    seed=SEED,
    learning_rate=3e-4,
    min_learning_rate=3e-6,
    weight_decay=1e-4,
    warmup_epochs=1,
    grad_accum_steps=1,
    gradient_clip=5.0,
    amp=True,
    projection_dim=128,
    hidden_dim=512,
    temperature=0.20,
    save_every=1,
).validate()

pd.Series({
    "method": config.method,
    "model": config.yolo_model,
    "epochs": config.epochs,
    "training images": len(train_images),
    "batch per GPU": config.batch_size,
    "source images per DDP step": config.batch_size * max(1, GPU_COUNT),
    "image size": config.image_size,
    "temperature": config.temperature,
    "mixed precision": config.amp,
})

## 4. Inspect the augmented views

The library applies random resized cropping, horizontal flipping, color jitter, grayscale conversion, blur, tensor conversion, and ImageNet normalization. Both views come from the same source image but receive independent random transformations.

The scene should stay recognizable across each pair. If crops routinely remove every player, raise the minimum crop scale.

In [ ]:
preview_dataset = UnlabeledImageDataset(train_images, build_transform(config))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

def display_tensor(tensor):
    return (tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)

fig, axes = plt.subplots(4, 2, figsize=(8, 15))
for row in range(4):
    first_view, second_view = preview_dataset[row]
    axes[row, 0].imshow(display_tensor(first_view))
    axes[row, 1].imshow(display_tensor(second_view))
    axes[row, 0].set_ylabel(f"Image {row + 1}")
    for axis in axes[row]:
        axis.set_xticks([])
        axis.set_yticks([])
axes[0, 0].set_title("View A")
axes[0, 1].set_title("View B")
plt.tight_layout()
plt.show()

## 5. Train on T4 x2

The launcher writes the configuration to the output directory and starts one process per visible
GPU. The progress bar reports running loss, learning rate, and EMA momentum. Checkpoints,
history, and the updated YOLO12 model land under `{OUTPUT_DIR}`.

In [ ]:
training_result = launch_distributed_pretrain(
    config,
    num_processes=max(1, GPU_COUNT),
    config_path=OUTPUT_DIR / "simclr_yolo12_config.yaml",
    check=True,
)

pd.Series({
    "completed": training_result.succeeded,
    "seconds": round(training_result.seconds, 1),
    "output directory": str(training_result.output_dir),
    "configuration": str(training_result.config_path),
})

## 6. Review the training loss

In [ ]:
history = pd.read_csv(OUTPUT_DIR / "history.csv")
display(history.round(6))

fig, axis = plt.subplots(figsize=(9, 5))
sns.lineplot(data=history, x="epoch", y="loss", marker="o", linewidth=2.5, ax=axis)
axis.set_title("SimCLR training loss on YOLO12")
axis.set_xlabel("Epoch")
axis.set_ylabel("NT-Xent loss")
axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

A falling loss shows the objective is being optimized. It is **not** a detection metric; only a labelled evaluation can tell you whether detection improved.

In [ ]:
manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text())
YOLO_CHECKPOINT = Path(manifest["outputs"]["yolo_checkpoint"])
SSL_CHECKPOINT = Path(manifest["outputs"]["ssl_checkpoint"])

pd.Series({
    "initialization": manifest["initialization"],
    "unlabeled images": manifest["unlabeled_images"],
    "world size": manifest["world_size"],
    "effective batch": manifest["effective_batch_size"],
    "best loss": manifest["best_loss"],
    "YOLO checkpoint": str(YOLO_CHECKPOINT),
    "full SSL checkpoint": str(SSL_CHECKPOINT),
})

## 7. Transfer the backbone into a YOLO12 detector

`transfer_ssl_backbone_to_yolo` copies **only** the online/student encoder into a fresh YOLO12
detector. Projection heads, predictors, target networks, the optimizer, the scheduler, and the
gradient scaler are all discarded.

The reported coverage should be 100%. A lower number means the YOLO12 scale used here does not
match the one used during pretraining.

In [ ]:
transfer = transfer_ssl_backbone_to_yolo(
    SSL_CHECKPOINT,
    OUTPUT_DIR / "simclr_yolo12_backbone.pt",
    yolo_model="yolo12n.yaml",
    minimum_coverage=0.95,
)

pd.Series({
    "SSL method": transfer.ssl_method,
    "source model": transfer.source_model,
    "encoder prefix": transfer.encoder_prefix,
    "loaded keys": f"{transfer.loaded_keys} / {transfer.total_backbone_keys}",
    "coverage": f"{transfer.coverage:.1%}",
    "detector checkpoint": str(transfer.detector_checkpoint),
    "transfer report": str(transfer.report_json),
})

The resulting checkpoint is a normal Ultralytics model. Fine-tune it on the labelled split to
find out whether pretraining helped:

```python
YOLO(str(transfer.detector_checkpoint)).train(
    data=str(DATASET_ROOT / "data.yaml"), epochs=50, imgsz=640
)
```

Compare that against a `yolo12n.yaml` baseline trained with identical settings. Without the
baseline the comparison says nothing.

## 8. Try an exercise

1. Re-run with `yolo12s.yaml` and compare the final loss. Larger backbones are not automatically
   better at a fixed epoch budget.
2. Halve `PER_GPU_BATCH_SIZE` and set `grad_accum_steps=2`. Note which quantity this keeps fixed
   and which it does not.
3. Train a `yolo12n.yaml` detector from scratch and a second from `simclr_yolo12_backbone.pt`,
   then evaluate both with `ssldet evaluate` on the test split.

## Practical checks

- Out-of-memory usually means the per-GPU batch is too large. Drop it from 32 to 16.
- Missing dataset paths mean the dataset was not attached through **Add Input**.
- Use `yolo12n.yaml`, never `yolov12n.yaml`.
- A coverage warning at transfer time means the YOLO12 scale changed between pretraining and
  transfer.